# IDR and Motif Workflow

Set `fasta_path` and optionally `record_id`, then run the cells top to bottom.

In [ ]:
import os
import sys

repo_root = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from Bio import SeqIO
from aaseq.annotations import annotate_functional_motifs
from aaseq.disorder import analyze_idr
from aaseq.motifs import get_fuzzy_repeats, getrep
from aaseq.statistics import motif_enrichment
from aaseq.tracks import motif_track_dataframe
from aaseq.visualization import plot_idr_profile, plot_motif_tracks

fasta_path = "examples/ncbi/example_proteins.fasta"
record_id = None  # e.g. "NP_000000.1"; None uses the first record
min_len, max_len, min_reps = 3, 10, 2

In [ ]:
records = list(SeqIO.parse(fasta_path, "fasta"))
record = next((r for r in records if record_id is None or r.id == record_id), None)
if record is None:
    raise ValueError(f"Record {record_id!r} not found")
sequence = str(record.seq).replace("\n", "").strip().upper()
record.id, len(sequence)

In [ ]:
exact_repeats = getrep(sequence, min_len, max_len, min_reps)
fuzzy_repeats = get_fuzzy_repeats(sequence, min_len, max_len, min_reps, max_mismatches=1)
sorted(exact_repeats.items(), key=lambda item: -item[1])[:20], sorted(fuzzy_repeats.items(), key=lambda item: -item[1])[:20]

In [ ]:
stats = motif_enrichment(sequence, min_len, max_len, min_reps, num_shuffles=200, seed=1, observed_counts=exact_repeats)
stats.head(20)

In [ ]:
scores, regions, summary = analyze_idr(sequence)
summary, regions.head()

In [ ]:
plot_idr_profile(scores, record.id, regions)

In [ ]:
tracks = motif_track_dataframe(record.id, sequence, min_len=min_len, max_len=max_len, min_reps=min_reps, top_n=20)
tracks.head(30)

In [ ]:
plot_motif_tracks(tracks, len(sequence), record.id)

In [ ]:
annotations = annotate_functional_motifs(sequence)
annotations.head(50)